In [1]:
from pathlib import Path

import automech
import cantera
import polars as pl
from automech.species import Species
from cantera.ck2yaml import Parser
from project_utilities import p_, util

tag = "Z_mess_v5"
file = util.notebook_file() if util.is_notebook() else __file__
root_path = Path("..")
chemkin_path = p_.chemkin(root_path)

In [2]:
tag0 = "Z_mess_v4"

# 2. Read in AVC species set with thermochemistry
mech_file0 = p_.calculated_mechanism(tag0, "json", p_.data(root_path))
mech0 = automech.io.read(mech_file0)
spc_df0 = mech0.species

In [3]:
def select_cho_species(spc_df: pl.DataFrame) -> pl.DataFrame:
    all_symbols = {f.name for f in spc_df.schema["formula"].fields}  # ty:ignore[unresolved-attribute]
    included_symbols = {"C", "H", "O"}
    excluded_symbols = all_symbols - included_symbols

    return spc_df.filter(
        pl.all_horizontal(
            [
                pl.col(Species.formula).struct.field(s).fill_null(0) == 0
                for s in excluded_symbols
            ]
        )
    )


def racemize(spc_df: pl.DataFrame) -> pl.DataFrame:
    return spc_df.with_columns(pl.col(Species.name).str.replace(r"[01]$", "R")).unique(
        subset=Species.name
    )


failed_species = [
    "C5H7(1315)",
    "S(1402)rR",
    "S(1336)rR",
    "S(1641)ze",
    "S(1633)",
    "S(1629)",
    "S(1641)ez",
    "S(1406)rR",
    "S(1250)e",
    "C5H8O(837)z",
    "S(1562)rsR",
    "S(1445)z",
    "S(1421)ze",
    "S(1511)e",
    "S(1492)rsR",
    "S(1583)z",
    "S(1344)z",
    "S(163)zz",
    "S(1411)rR",
    "S(1485)e",
    "S(1335)rR",
    "S(1477)",
    "S(1615)",
    "C5H9O(856)",
    "S(1344)e",
    "S(1644)rsR",
    "S(1641)ee",
    "S(1445)e",
    "S(1673)zz",
    "S(1635)rR",
    "S(1492)rrR",
    "S(1673)ez",
    "S(1370)",
    "S(1388)",
    "S(1588)z",
    "S(1348)rR",
    "S(1502)eeee",
    "S(1613)z",
    "C4H7(1434)",
    "S(1373)rsR",
    "S(1579)rR",
    "S(1644)rrR",
    "S(1588)e",
    "S(1453)eee",
    "S(1613)e",
    "S(1599)",
    "S(1583)e",
    "S(1624)",
    "S(1374)rR",
    "S(1640)zz",
    "S(1641)zz",
    "S(1250)z",
    "S(1532)erR",
    "S(1504)rsR",
    "S(1385)z",
    "S(1562)rrR",
    "S(1500)rsR",
    "S(1569)",
]


def drop_failed_species(spc_df: pl.DataFrame) -> pl.DataFrame:
    return spc_df.filter(~pl.col(Species.name).is_in(failed_species))

In [4]:
tag1 = "cyclopentene_species_expanded"

# 1a. Read in ASC species set
spc_file1 = p_.data(root_path) / f"{tag1}.json"
spc_df1 = pl.read_json(spc_file1)
spc_df1 = select_cho_species(spc_df1)
spc_df1 = racemize(spc_df1)
spc_df1 = drop_failed_species(spc_df1)

# 1b. Read in ASC thermochemistry
therm_file1 = p_.chemkin(root_path) / f"{tag1}.dat"
spc_df1 = automech.io.chemkin.read.thermo(
    therm_file1, spc_df=spc_df1, racemize=True, complete=True
)
assert spc_df1 is not None

# 1c. Remove species that were already included
spc_df1 = spc_df1.join(spc_df0, on=Species.name, how="anti")

In [5]:
# 3. Combine species sets and create new mechanism
mech = mech0.model_copy()
mech.species = pl.concat([spc_df0, spc_df1], how="vertical_relaxed")

calc_mech_json = p_.calculated_mechanism(tag, "json", p_.data(root_path))
print(f" - Calculated: {calc_mech_json}")
automech.io.write(mech, calc_mech_json)

calc_mech_ckin = p_.calculated_mechanism(tag, "dat", p_.chemkin(root_path))
print(f" - Calculated: {calc_mech_ckin}")
automech.io.chemkin.write.mechanism(mech, calc_mech_ckin)

calc_mech_yaml = p_.calculated_mechanism(tag, "yaml", path=p_.cantera(root_path))
print(f" - Calculated: {calc_mech_yaml}")

print("Converting ChemKin mechanism to Cantera YAML...")
Parser.convert_mech(calc_mech_ckin, out_name=calc_mech_yaml)

print("Validating Cantera model...")
cantera.Solution(calc_mech_yaml)

 - Calculated: ../data/Z_mess_v5_calc.json
 - Calculated: ../data/chemkin/Z_mess_v5_calc.dat
 - Calculated: ../data/cantera/Z_mess_v5_calc.yaml
Converting ChemKin mechanism to Cantera YAML...
Wrote YAML mechanism file to '../data/cantera/Z_mess_v5_calc.yaml'.
Mechanism contains 952 species and 140 reactions.
Validating Cantera model...
